In [32]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV

M = 10 # number of levels in the book, paper and data use 10
# We define all the needed functions below. Later, you can see example usage.
def load_and_prepare(filepath):
    """
    Load order book CSV, parse timestamps, sort chronologically, and reset index.
    """
    df = pd.read_csv(filepath)
    df["ts_event"] = pd.to_datetime(df["ts_event"], utc=True)
    df.sort_values("ts_event", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

def compute_ofi_features(df, M):
    """
    Compute each OFI and adds them to the df.
    """
    for m in range(M):
        bid_price = f"bid_px_0{m}"
        bid_size = f"bid_sz_0{m}"
        ask_price = f"ask_px_0{m}"
        ask_size = f"ask_sz_0{m}"

        bid_price_prev = bid_price + "_prev"
        bid_size_prev = bid_size + "_prev"
        ask_price_prev = ask_price + "_prev"
        ask_size_prev = ask_size + "_prev"

        # Order book snapshot
        df[bid_price_prev] = df[bid_price].shift(1)
        df[bid_size_prev] = df[bid_size].shift(1)
        df[ask_price_prev] = df[ask_price].shift(1)
        df[ask_size_prev] = df[ask_size].shift(1)

        # Bid-side OF
        df[f"OFb_{m}"] = np.where(
            df[bid_price] > df[bid_price_prev],
            df[bid_size],
            np.where(
                df[bid_price] < df[bid_price_prev],
                -df[bid_size],
                df[bid_size] - df[bid_size_prev]
            )
        )

        # Ask-side OF
        df[f"OFa_{m}"] = np.where(
            df[ask_price] < df[ask_price_prev],
            df[ask_size],
            np.where(
                df[ask_price] > df[ask_price_prev],
                -df[ask_size],
                df[ask_size] - df[ask_size_prev]
            )
        )

        # Net order flow imbalance
        df[f"OFI_{m}"] = df[f"OFb_{m}"] - df[f"OFa_{m}"]

def deep_ofi(df, h, t, level):
    """
    Total OFI at a given book level over interval (t-h, t].
    """
    mask = (df["ts_event"] > (t - h)) & (df["ts_event"] <= t)
    return df.loc[mask, f"OFI_{level}"].sum()
#Best-Level OFI
def best_ofi(df, h, t):
    """
    Best-level OFI over interval (t-h, t].
    """
    return deep_ofi(df, h, t, 0)

def compute_Q(df, h, t, M):
    """
    Compute normalization factor Q for OFI over interval (t-h, t].
    """
    mask = (df["ts_event"] > (t - h)) & (df["ts_event"] <= t)
    N = mask.sum()
    if N == 0:
        return np.nan
    total = sum(
        df.loc[mask, f"bid_sz_0{m}"].sum() + df.loc[mask, f"ask_sz_0{m}"].sum()
        for m in range(M)
    )
    return total / (2 * M * N)
#Multi-Level OFI
def multi_level_ofi(df, h, t, M):
    """
    Compute normalized multi-level OFI vector at time t.
    """
    Q = compute_Q(df, h, t, M)
    return np.array([deep_ofi(df, h, t, m) / Q for m in range(M)])

def fit_pca_weights(X):
    """
    Fit PCA to multi-level OFI matrix and returns normalized first component.
    """
    pca = PCA(n_components=1).fit(X)
    w = pca.components_[0]
    return w / np.sum(np.abs(w))
#Integrated OFI
def integrated_ofi(df, h, t, w):
    """
    Project multi-level OFI at time t onto PCA-derived weight w.
    """
    vec = multi_level_ofi(df, h, t, len(w))
    return float(np.dot(w, vec))
def log_return(df, h, t):
    """
    Compute log return over (t-h, t]
    """
    mask = (df["ts_event"] > (t - h)) & (df["ts_event"] <= t)
    df_sub = df.loc[mask]
    if df_sub.empty:
        return np.nan
    first, last = df_sub.iloc[0], df_sub.iloc[-1]
    p0 = first["ask_px_00"] + first["bid_px_00"]
    p1 = last["ask_px_00"] + last["bid_px_00"]
    return np.log(p1) - np.log(p0)
# The Lasso fit would be over given timestamps, could be over all timestamps.
def fit_cross_asset_ofi(df, h, timestamps, target, tickers):
    """
    Fit LASSO regression of target's log return on best-level OFI of peer tickers.
    """
    peers = [sym for sym in tickers if sym != target]
    if not peers:
        raise ValueError("No peers available for cross-asset OFI!")

    X, Y = [], []
    for t in timestamps:
        y = log_return(df, h, t)
        feat = [best_ofi(df, h, t) for _ in peers]
        Y.append(y)
        X.append(feat)

    X = np.asarray(X)
    Y = np.asarray(Y)
    lasso = LassoCV(cv=5).fit(X, Y)
    return peers, lasso.coef_, lasso.intercept_

#Cross-Asset OFI. Since our sample file only contains a single symbol,
# true cross-asset OFI (Eqn 5 in the paper) cannot be computed.
def cross_asset_ofi_at(df, h, t, target,tickers):
    """
    Compute instantaneous cross-asset OFI: weighted sum of peer OFIs at time t.
    """
    feats = np.array([best_ofi(df, h, t) for _ in peers])
    return float(np.dot(betas, feats))


In [31]:
# Example usage

# 1) Load & prepare data
df = load_and_prepare("first_25000_rows.csv")

# 2) Compute OFI features for all M levels
compute_ofi_features(df, M)

# 3) Pick the 100th event and a 30-second window (for instance)
t = df.ts_event.iloc[100]
h = pd.Timedelta(seconds=30)

# 4) Compute best-level and multi-level OFI at (t-h, t]
best = best_ofi(df, h, t)
multi = multi_level_ofi(df, h, t, M)

# 5) Build OFI matrix (dropping any rows with NaNs)
ofi_cols    = [f"OFI_{m}" for m in range(M)]
ofi_matrix  = df[ofi_cols].dropna().to_numpy()

# 6) Fit PCA to get weights w
w = fit_pca_weights(ofi_matrix)

# 7) Compute the integrated OFI
integrated = integrated_ofi(df, h, t, w)

# 8) Display results
print(f"Timestamp:           {t}")
print(f"Window length:       {h}")
print(f"Best-level OFI:      {best}")
print(f"Multi-level OFI:     {multi}")
print(f"PCA weights (w):     {w}")
print(f"Integrated OFI (I):  {integrated}")

Timestamp:           2024-10-21 11:56:34.567796323+00:00
Window length:       0 days 00:00:30
Best-level OFI:      233.0
Multi-level OFI:     [ 2.10140876 -5.23097458 15.82820758 -2.57039268  5.79014773 -5.51958008
  2.23669258  5.68192067 -5.7360342   9.4788867 ]
PCA weights (w):     [0.03056411 0.08027476 0.13937689 0.16486505 0.1365606  0.11653604
 0.08897391 0.0915125  0.0796825  0.07165364]
Integrated OFI (I):  2.515216206087855
